# Assignment 7 – Image Sharpening

**Objective:** Sharpen an input image using:
1. Laplacian filter
2. Sobel gradient operator
3. Canny edge detector
4. Unsharp masking using a Gaussian filter of size **5×5** and **sigma = 3**

Run the notebook cells from top to bottom. Place your input image in the same folder as this notebook, or enter its full path when prompted.

In [ ]:
# Install the required libraries (run once if they are not already installed)
%pip install opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV version:", cv2.__version__)

## 1. Load the input image

Enter the image filename or full path when prompted. The program reads the image in colour and converts it to grayscale for the filtering operations.

In [ ]:
image_path = input("Enter image path (for example: input.jpg): ").strip()

image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(
        f"Could not read '{image_path}'. Check the filename/path and try again."
    )

# OpenCV loads colour images in BGR format; convert to RGB for Matplotlib.
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print("Image loaded successfully.")
print("Image dimensions:", image.shape)

plt.figure(figsize=(7, 5))
plt.imshow(image_rgb)
plt.title("Original Input Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(7, 5))
plt.imshow(gray, cmap="gray")
plt.title("Grayscale Input Image")
plt.axis("off")
plt.show()

## 2. Sharpening using the Laplacian filter

The Laplacian filter detects rapid intensity changes (second-order derivatives). These changes highlight edges and fine details. We subtract a scaled Laplacian response from the original image to enhance edges.

In [ ]:
# Calculate the Laplacian response
laplacian = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)

# Convert the response to an 8-bit image for display
laplacian_abs = cv2.convertScaleAbs(laplacian)

# Sharpen by subtracting the Laplacian response
laplacian_sharpened = cv2.addWeighted(
    gray, 1.0, laplacian_abs, -0.7, 0
)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(laplacian_abs, cmap="gray")
plt.title("Laplacian Response")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(laplacian_sharpened, cmap="gray")
plt.title("Laplacian Sharpened")
plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Sharpening using the Sobel gradient operator

Sobel filters estimate horizontal and vertical intensity gradients. Combining these gradients gives an edge-strength image. We add a scaled edge-strength image to the original grayscale image to emphasize edges.

In [ ]:
# Compute horizontal and vertical Sobel gradients
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

# Gradient magnitude represents edge strength
sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
sobel_edges = cv2.convertScaleAbs(sobel_magnitude)

# Add a controlled amount of the gradient magnitude to sharpen
sobel_sharpened = cv2.addWeighted(gray, 1.0, sobel_edges, 0.35, 0)
sobel_sharpened = np.clip(sobel_sharpened, 0, 255).astype(np.uint8)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(sobel_edges, cmap="gray")
plt.title("Sobel Gradient Magnitude")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(sobel_sharpened, cmap="gray")
plt.title("Sobel Sharpened")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Canny edge detector

Canny detects edges using smoothing, gradient calculation, non-maximum suppression and hysteresis thresholding. The resulting edge map is added in a controlled amount to the original image to emphasize detected edges.

In [ ]:
# Canny edge detection
canny_edges = cv2.Canny(gray, threshold1=100, threshold2=200)

# Add detected edges to the original image
canny_sharpened = cv2.addWeighted(gray, 1.0, canny_edges, 0.4, 0)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(canny_edges, cmap="gray")
plt.title("Canny Edge Map")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(canny_sharpened, cmap="gray")
plt.title("Canny Edge Enhanced")
plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Sharpening using Unsharp Masking

Unsharp masking creates a blurred version of the image, subtracts it from the original to obtain a detail mask, and adds that mask back to the original.

**Assignment requirement:** Gaussian kernel size = **5×5**, sigma = **3**.

In [ ]:
# Blur using the exact Gaussian settings specified in the assignment
gaussian_blurred = cv2.GaussianBlur(
    gray, (5, 5), sigmaX=3, sigmaY=3
)

# Create the detail mask and add it back to the original
detail_mask = cv2.subtract(gray, gaussian_blurred)
unsharp_masked = cv2.addWeighted(
    gray, 1.5, gaussian_blurred, -0.5, 0
)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gaussian_blurred, cmap="gray")
plt.title("Gaussian Blur (5×5, sigma=3)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(unsharp_masked, cmap="gray")
plt.title("Unsharp Masked")
plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Compare all sharpening results

The following figure displays the original image and the results produced by each method.

In [ ]:
results = [
    ("Original", gray),
    ("Laplacian", laplacian_sharpened),
    ("Sobel", sobel_sharpened),
    ("Canny enhanced", canny_sharpened),
    ("Unsharp masking", unsharp_masked),
]

plt.figure(figsize=(15, 9))
for index, (title, result) in enumerate(results, start=1):
    plt.subplot(2, 3, index)
    plt.imshow(result, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 7. Save the output images

This cell saves the processed images in an `assignment7_outputs` folder next to the input image.

In [ ]:
output_dir = Path(image_path).resolve().parent / "assignment7_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "original_grayscale.png": gray,
    "laplacian_response.png": laplacian_abs,
    "laplacian_sharpened.png": laplacian_sharpened,
    "sobel_gradient.png": sobel_edges,
    "sobel_sharpened.png": sobel_sharpened,
    "canny_edges.png": canny_edges,
    "canny_sharpened.png": canny_sharpened,
    "gaussian_blurred_5x5_sigma3.png": gaussian_blurred,
    "unsharp_masked.png": unsharp_masked,
}

for filename, result in outputs.items():
    cv2.imwrite(str(output_dir / filename), result)

print("All output images saved in:", output_dir)

## 8. Conclusion

The input image was sharpened using the Laplacian filter, Sobel gradient operator, Canny edge detector and Unsharp Masking. The Laplacian and gradient-based methods emphasize edges, while Unsharp Masking enhances fine details by adding a detail mask derived from a Gaussian-blurred image. The Gaussian blur used for Unsharp Masking has the required 5×5 kernel and sigma of 3.